# Build a Numerai v6 model

Train a small example model, inspect validation predictions, and prepare a live predictions file. The dataset keys and columns in the configuration cell must be filled from the approved Numerai v6 release manifest before running this notebook.


In [ ]:
!pip install -q numerapi 'pandas>=2.3,<3' pyarrow lightgbm scikit-learn


## 1. Select the approved dataset

Use the published Classic tournament dataset keys. This notebook stops until the release manifest is confirmed.


In [ ]:
from numerapi import NumerAPI
import pandas as pd
import lightgbm as lgb

TRAIN_DATASET = None
VALIDATION_DATASET = None
LIVE_DATASET = None
TARGET_COLUMN = None
ID_COLUMN = None

config = {
    "TRAIN_DATASET": TRAIN_DATASET,
    "VALIDATION_DATASET": VALIDATION_DATASET,
    "LIVE_DATASET": LIVE_DATASET,
    "TARGET_COLUMN": TARGET_COLUMN,
    "ID_COLUMN": ID_COLUMN,
}
missing = [name for name, value in config.items() if not value]
if missing:
    raise ValueError(f"Set approved Numerai v6 values before running: {missing}")

napi = NumerAPI()
available = set(napi.list_datasets())
for dataset in (TRAIN_DATASET, VALIDATION_DATASET, LIVE_DATASET):
    if dataset not in available:
        raise ValueError(f"Dataset is not listed by NumerAPI: {dataset}")


In [ ]:
for dataset in (TRAIN_DATASET, VALIDATION_DATASET, LIVE_DATASET):
    napi.download_dataset(dataset)

import pyarrow as pa
import pyarrow.parquet as pq

train_schema = pq.read_schema(TRAIN_DATASET)
validation_schema = pq.read_schema(VALIDATION_DATASET)
live_schema = pq.read_schema(LIVE_DATASET)
for name, schema in (("train", train_schema), ("validation", validation_schema)):
    if TARGET_COLUMN not in schema.names:
        raise ValueError(f"{TARGET_COLUMN} is missing from {name}")
if ID_COLUMN not in live_schema.names:
    raise ValueError(f"{ID_COLUMN} is missing from live")

shared_columns = set(validation_schema.names) & set(live_schema.names)
feature_columns = [
    name for name in train_schema.names
    if name.startswith("feature_")
    and name in shared_columns
    and pa.types.is_floating(train_schema.field(name).type)
][:50]
if not feature_columns:
    raise ValueError("No shared numeric feature columns found in the approved files")

train = pd.read_parquet(TRAIN_DATASET, columns=feature_columns + [TARGET_COLUMN])
validation = pd.read_parquet(VALIDATION_DATASET, columns=feature_columns + [TARGET_COLUMN])
live = pd.read_parquet(LIVE_DATASET, columns=feature_columns + [ID_COLUMN])
print("Train:", train.shape, "Validation:", validation.shape, "Live:", live.shape)


## 2. Train and inspect a starter model

Select numeric feature columns shared by train, validation, and live. Use a bounded training sample so the example fits in a typical Colab session. This is a learning example, not an optimized tournament model.


In [ ]:
train_rows = train.dropna(subset=[TARGET_COLUMN])
if train_rows.empty:
    raise ValueError("The approved training file has no mature target rows")
train_sample = train_rows.sample(n=min(100_000, len(train_rows)), random_state=7)
model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31)
model.fit(train_sample[feature_columns], train_sample[TARGET_COLUMN])


In [ ]:
validation_rows = validation.dropna(subset=[TARGET_COLUMN])
if validation_rows.empty:
    raise ValueError("The approved validation file has no mature target rows")
validation_sample = validation_rows.sample(
    n=min(50_000, len(validation_rows)), random_state=7
)
validation_predictions = model.predict(validation_sample[feature_columns])
print(pd.Series(validation_predictions, name="prediction").describe())


## 3. Prepare live predictions

Save a file for inspection. The tournament submission workflow is documented separately.


In [ ]:
live_predictions = model.predict(live[feature_columns])
submission = live[[ID_COLUMN]].copy()
submission["prediction"] = pd.Series(live_predictions, index=live.index).rank(pct=True)
submission.to_csv("numerai_v6_example_predictions.csv", index=False)
submission.head()
